In [ ]:
from datetime import datetime
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================
# ADLS OAuth Configuration
# Replace YOUR_CLIENT_SECRET
# ==========================
for acct in ["adlsdevvbsrc01","adlsdevvbstd01"]:
    spark.conf.set(f"fs.azure.account.auth.type.{acct}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{acct}.dfs.core.windows.net",
                   "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{acct}.dfs.core.windows.net",
                   "2c5d85d1-759d-4e41-9703-905b120f74f5")
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{acct}.dfs.core.windows.net",
                   "YOUR_CLIENT_SECRET")
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{acct}.dfs.core.windows.net",
                   "https://login.microsoftonline.com/7cab1251-58e4-4efb-8936-ae9b0cb98fce/oauth2/token")

source_root="abfss://output@adlsdevvbsrc01.dfs.core.windows.net/dim_scd2"
delta_path="abfss://output@adlsdevvbstd01.dfs.core.windows.net/payment_delta_scd2"
table_name="dim_order_payments_scd2"

# Latest folder
y=max(f.name.rstrip("/") for f in dbutils.fs.ls(source_root))
m=max(f.name.rstrip("/") for f in dbutils.fs.ls(f"{source_root}/{y}"))
d=max(f.name.rstrip("/") for f in dbutils.fs.ls(f"{source_root}/{y}/{m}"))

latest_path=f"{source_root}/{y}/{m}/{d}/*.parquet"
file_date=f"{y}-{m}-{d}"

raw_df=spark.read.parquet(latest_path).withColumn("updated_at",current_timestamp())

w=Window.partitionBy("order_id","payment_sequential").orderBy(col("updated_at").desc())
dedup_df=raw_df.withColumn("rn",row_number().over(w)).filter("rn=1").drop("rn")

incoming_df=(dedup_df
 .withColumn("payment_dim_id",expr("uuid()"))
 .withColumn("is_active",lit(1))
 .withColumn("start_date",current_date())
 .withColumn("end_date",lit(None).cast("date"))
 .withColumn("file_date",lit(file_date))
 .withColumn("ingest_date",current_date())
 .withColumn("updated_at",current_timestamp()))

if not DeltaTable.isDeltaTable(spark,delta_path):
    incoming_df.write.format("delta").mode("overwrite").option("mergeSchema","true").save(delta_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{delta_path}'")
else:
    dt=DeltaTable.forPath(spark,delta_path)
    active=dt.toDF().filter(col("is_active")==1)

    changes=(incoming_df.alias("s")
      .join(active.alias("t"),
            (col("s.order_id")==col("t.order_id")) &
            (col("s.payment_sequential")==col("t.payment_sequential")),
            "left")
      .filter(
          col("t.payment_type").isNull() |
          (col("t.payment_type")!=col("s.payment_type")) |
          (col("t.payment_installments")!=col("s.payment_installments")) |
          (col("t.payment_value")!=col("s.payment_value"))
      ).select("s.*"))

    wc=Window.partitionBy("order_id","payment_sequential").orderBy(col("updated_at").desc())
    changes=changes.withColumn("rn",row_number().over(wc)).filter("rn=1").drop("rn")

    (dt.alias("t")
      .merge(changes.alias("s"),
             "t.order_id=s.order_id AND t.payment_sequential=s.payment_sequential AND t.is_active=1")
      .whenMatchedUpdate(set={
          "is_active":"0",
          "end_date":"current_date()",
          "updated_at":"current_timestamp()"
      }).execute())

    (dt.alias("t")
      .merge(changes.alias("s"),
             "t.order_id=s.order_id AND t.payment_sequential=s.payment_sequential AND t.is_active=0")
      .whenNotMatchedInsert(values={c:col(f"s.{c}") for c in incoming_df.columns})
      .execute())

display(spark.read.format("delta").load(delta_path))


In [ ]:
# === Replace mount paths with direct ADLS paths ===
source_root="abfss://output@adlsdevvbsrc01.dfs.core.windows.net/dim_scd2"
delta_path="abfss://output@adlsdevvbstd01.dfs.core.windows.net/payment_delta_scd2"

folders = dbutils.fs.ls(source_root)
latest_year=max([f.name.replace('/','') for f in folders])
folders=dbutils.fs.ls(f"{source_root}/{latest_year}")
latest_month=max([f.name.replace('/','') for f in folders])
folders=dbutils.fs.ls(f"{source_root}/{latest_year}/{latest_month}")
latest_day=max([f.name.replace('/','') for f in folders])

latest_path=f"{source_root}/{latest_year}/{latest_month}/{latest_day}/*.parquet"
file_date=f"{latest_year}-{latest_month}-{latest_day}"

# Paste your existing Step 2 through Step 7 SCD2 merge logic here unchanged.
# Only the paths need to be updated:
# /mnt/source/dim_scd2 -> source_root
# /mnt/curated/payment_delta_scd2 -> delta_path
